# 02 · Table I: selection strategies (cache only, about 10 min on a GPU)

Every strategy scores the **identical** K = 8 candidates of the main draw
through the same vote, so differences come from the per-candidate score alone.

| row | how it is computed |
|---|---|
| Majority (arbitrary / log-prob tie-break) | `selection_hits(..., "majority" / "majority_lp")` |
| Token-confidence weighted voting | weights `exp(mean token log-prob)` |
| LiLaVe-style GBT | XGBoost on the same 4,608 + 5 features, 10 seeds |
| **LVPP** | 1.18M-parameter verifier, 10 training seeds |

Also produced here: the significance tests quoted in Section V-A, the
single-best-candidate comparison, and the **stand-alone classifier
accuracies** on the verifier pool. (The three additional candidate draws
quoted in Section V-A, +3.16 / +4.27 / +3.41 points, are separate captures:
rerun notebook 00 with a new seed and point `PATHS` at it.)

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gbt] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
from lvpp.caches import load_pool_A, load_dict_B
from lvpp.capture import MULTI
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier, find_logprob_col
from lvpp.baselines import token_confidence, gbt_scores
from lvpp.stats import paired, mean_sd

P, Z, y, _ = load_pool_A(PATHS["main_pool"])
te = load_dict_B(PATHS["main_test"])
print(f"pool {len(y)} candidates (pos-rate {y.mean():.3f}) | test {te.nq} questions, {len(te.y)} candidates")
h_maj = selection_hits(te, "majority"); h_lp = selection_hits(te, "majority_lp")
conf = token_confidence(te, "exp"); h_conf = selection_hits(te, "weighted", scores=conf)

In [ ]:
SEEDS = list(range(10))
ver_acc, ver_auc, gbt_acc, gbt_auc, H, p_vs_maj = [], [], [], [], [], []
for s in SEEDS:
    v = fit_verifier(P, Z, y, seed=s)
    sc = v.score(te.P, te.Z)
    h = selection_hits(te, "weighted", scores=sc); H.append(h)
    ver_acc.append(100 * h.mean()); ver_auc.append(roc_auc_score(te.y.numpy(), sc))
    p_vs_maj.append(paired(h, h_maj)["p"])
    try:
        sg = gbt_scores(P, Z, y, te.P, te.Z, seed=s)
        gbt_acc.append(100 * selection_hits(te, "weighted", scores=sg).mean())
        gbt_auc.append(roc_auc_score(te.y.numpy(), sg))
    except Exception as e:  # XGBoost missing or crashing: report LVPP anyway
        print(f"   GBT skipped for seed {s}: {type(e).__name__}: {str(e)[:120]}")
        gbt_acc.append(np.nan); gbt_auc.append(np.nan)
    print(f"seed {s}: LVPP {ver_acc[-1]:.2f} (AUC {ver_auc[-1]:.4f}) | GBT {gbt_acc[-1]:.2f} (AUC {gbt_auc[-1]:.4f})")

In [ ]:
m, sd_pop = mean_sd(ver_acc); _, sd_samp = mean_sd(ver_acc, ddof=1)
rows = {
    "majority (arbitrary)": 100 * h_maj.mean(),
    "majority (log-prob tie-break)": 100 * h_lp.mean(),
    "token-confidence weighted": 100 * h_conf.mean(),
    "token-confidence AUC": roc_auc_score(te.y.numpy(), conf),
    "GBT accuracy (10-seed mean)": None if np.all(np.isnan(gbt_acc)) else float(np.nanmean(gbt_acc)),
    "GBT AUC": None if np.all(np.isnan(gbt_auc)) else float(np.nanmean(gbt_auc)),
    "LVPP accuracy (10-seed mean)": m,
    "LVPP AUC": float(np.mean(ver_auc)),
    "LVPP gain over majority": m - 100 * h_maj.mean(),
    "worst-seed McNemar p vs majority": max(p_vs_maj),
}
EXPECTED = {"majority (arbitrary)": 74.91, "majority (log-prob tie-break)": 74.45,
            "token-confidence weighted": 74.45, "token-confidence AUC": 0.648,
            "GBT accuracy (10-seed mean)": 76.57, "GBT AUC": 0.874,
            "LVPP accuracy (10-seed mean)": 78.84, "LVPP AUC": 0.869,
            "LVPP gain over majority": 3.93, "worst-seed McNemar p vs majority": 1.3e-6}
df = report(rows, EXPECTED, list(EXPECTED))
print(f"\nLVPP sd over seeds: population {sd_pop:.2f}, sample {sd_samp:.2f}")
df.to_csv(f"{RESULTS}/table1.csv", index=False)

In [ ]:
#@title Other confidence variants and the best-confidence comparison (Section V-A)
best = None
for var in ["exp", "softmax", "zscore"]:
    w = token_confidence(te, var); h = selection_hits(te, "weighted", scores=w)
    print(f"confidence ({var:7s}) {100*h.mean():.2f}%")
    if best is None or h.mean() > best[1].mean(): best = (var, h)
r = paired(H[0], best[1])
print(f"\nLVPP seed 0 vs best confidence ({best[0]}): {r['diff']:+.2f} points, "
      f"95% CI [{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]  (paper: +3.79, [+2.27, +5.31])")

In [ ]:
#@title Single highest-scored candidate vs weighted voting
v0 = fit_verifier(P, Z, y, seed=0); sc0 = v0.score(te.P, te.Z)
h_w = selection_hits(te, "weighted", scores=sc0); h_s = selection_hits(te, "single_best", scores=sc0)
print(f"weighted vote {100*h_w.mean():.2f}% | single best {100*h_s.mean():.2f}% "
      f"(paper: single best is 1.3-4.3 points lower across generators)")

In [ ]:
#@title Stand-alone correctness classification on the verifier pool (Section V-A text)
from lvpp.probe import probe_accuracy, majority_class_rate
lp_col = find_logprob_col(Z.numpy())
acc_h, _ = probe_accuracy(P, y)
acc_lp, _ = probe_accuracy(Z[:, [lp_col]], y)
acc_hlp, _ = probe_accuracy(torch.cat([P, Z[:, [lp_col]]], -1), y)
print(f"hidden states (3 layers)  {100*acc_h:.1f}%")
print(f"mean token log-prob only  {100*acc_lp:.1f}%")
print(f"hidden + log-prob         {100*acc_hlp:.1f}%")
print(f"majority-class rate       {100*majority_class_rate(y):.1f}%")
print("original run on this pool: 87.3 / 64.5 / 87.2 / 76.8")